In [ ]:
!pip install opencv-python scikit-image scikit-learn tqdm

In [4]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from skimage.feature import local_binary_pattern


# ==========================
# CONFIG
# ==========================

DATA_ROOT = r"e:\PythonFile\Project\Facial-Recognition\data\CASIA_faceAntisp"

IMG_SIZE = 64
P = 8
R = 1
METHOD = "uniform"
COLOR_SPACE = "ycrcb"

FPS = 25
WINDOW_SEC = 3
OVERLAP_SEC = 2

REAL = 1
ATTACK = 0

# Load Haar Cascade
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)


# ==========================
# LBP FEATURE
# ==========================
def extract_lbp_nonri_fast(gray, P=8, R=1):

    lbp = local_binary_pattern(gray, P, R, method="default").astype(np.uint8)

    bits = ((lbp[..., None] >> np.arange(P)) & 1).astype(np.uint8)

    shifted = np.roll(bits, -1, axis=2)
    transitions = np.sum(bits != shifted, axis=2)

    uniform_mask = transitions <= 2

    non_uniform_label = P*(P-1) + 2
    lbp_uniform = lbp.copy()
    lbp_uniform[~uniform_mask] = non_uniform_label

    n_bins = P*(P-1) + 3

    hist, _ = np.histogram(
        lbp_uniform.ravel(),
        bins=n_bins,
        range=(0, n_bins)
    )
    print(hist, hist.shape)
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)
    

    return hist


# ==========================
# FACE DETECTION + CROP
# ==========================

def detect_and_crop_face(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=5,
        minSize=(50, 50)
    )

    if len(faces) == 0:
        return None

    faces = sorted(faces, key=lambda x: x[2]*x[3], reverse=True)
    (x, y, w, h) = faces[0]

    face = frame[y:y+h, x:x+w]
    face = cv2.resize(face, (IMG_SIZE, IMG_SIZE))
    cv2.imshow("Face", face)   
    cv2.waitKey(1)        

    return face


# ==========================
# COLOR CONVERT
# ==========================

def convert_color(face):
    if COLOR_SPACE == "gray":
        img = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
        return [img]

    elif COLOR_SPACE == "rgb":
        img = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        return cv2.split(img)

    elif COLOR_SPACE == "hsv":
        img = cv2.cvtColor(face, cv2.COLOR_BGR2HSV)
        return cv2.split(img)

    elif COLOR_SPACE == "ycrcb":
        img = cv2.cvtColor(face, cv2.COLOR_BGR2YCrCb)
        return cv2.split(img)


# ==========================
# VIDEO → FRAME FEATURES
# ==========================

def extract_video_features(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    print("FPS của video là:", fps)
    features = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        face = detect_and_crop_face(frame)

        if face is None:
            continue

        channels = convert_color(face)

        channel_features = []
        for ch in channels:
            hist = extract_lbp_nonri_fast(ch)
            channel_features.extend(hist)
            print(channels, channels.shape)

        features.append(channel_features)

    cap.release()

    if len(features) == 0:
        return None

    return np.array(features)


# ==========================
# TEMPORAL WINDOW
# ==========================

def temporal_windows(features):
    window_size = int(FPS * WINDOW_SEC)
    overlap = int(FPS * OVERLAP_SEC)
    step = window_size - overlap

    windows = []
    for start in range(0, len(features) - window_size + 1, step):
        window = features[start:start+window_size]
        windows.append(np.mean(window, axis=0))

    return windows


def first_window(features):
    window_size = int(FPS * WINDOW_SEC)

    if len(features) < window_size:
        return None

    return np.mean(features[:window_size], axis=0)


# ==========================
# LABEL FUNCTION
# ==========================

def get_label(filename):
    name = filename.lower()

    if name.startswith("1") or name.startswith("2") or \
       name.startswith("hr_1"):
        return REAL
    else:
        return ATTACK


# ==========================
# LOAD DATA
# ==========================

def load_split(split="train"):
    root = os.path.join(DATA_ROOT, f"{split}_release")

    X = []
    y = []

    subjects = os.listdir(root)
    count = 0

    for subject in tqdm(subjects):
        subject_path = os.path.join(root, subject)

        for video in os.listdir(subject_path):
            if count == 1: break
            count += 1
            video_path = os.path.join(subject_path, video)

            label = get_label(video)

            frame_feats = extract_video_features(video_path)

            if frame_feats is None:
                continue

            if split == "train":
                windows = temporal_windows(frame_feats)
                for w in windows:
                    X.append(w)
                    y.append(label)
            else:
                feat = first_window(frame_feats)
                if feat is not None:
                    X.append(feat)
                    y.append(label)

    return np.array(X), np.array(y)


# ==========================
# TRAIN + TEST
# ==========================

print("Extracting training data...")
X_train, y_train = load_split("train")

print("Extracting testing data...")
X_test, y_test = load_split("test")


# ==========================
# SAVE FEATURES
# ==========================

save_path = f"E:/PythonFile/Project/Facial-Recognition/data/feature/casia_lbp_face_u_{COLOR_SPACE}_features.npz"

np.savez(
    save_path,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test
)

print("Saved features to:", save_path)

Extracting training data...


  0%|          | 0/20 [00:00<?, ?it/s]

FPS của video là: 25.0
[ 99  31   4   9  40   0   8  61   2   0   0   0  13   0  16  55  41   0
   0   0   0   0   0   0  17   0   0   0  89   0 107 102  10   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0  34   0   0   0   0   0
   0   0  48   0 288] (59,)


AttributeError: 'tuple' object has no attribute 'shape'